## There are 844 missing values in the borough column. I am going to use the NYC GeoSearch api to look up the missing borough by address or street.


In [ ]:
import time
import requests

missing_boroughs = clean_sample.loc[clean_sample["borough"].isna()].copy()

# House Number was not retained in clean_sample, but the original indexes remain.
missing_boroughs["house_number"] = sample.loc[missing_boroughs.index, "House Number"]

missing_boroughs["lookup_address"] = (
    missing_boroughs["house_number"].fillna("").str.strip()
    + " "
    + missing_boroughs["street_name"].fillna("").str.strip()
    + ", New York, NY"
).str.strip()

display(
    missing_boroughs[
        [
            "summons_number",
            "house_number",
            "street_name",
            "violation_precinct",
            "violation_county",
            "lookup_address",
        ]
    ].head(20)
)

,summons_number,house_number,street_name,violation_precinct,violation_county,lookup_address
7,5604629856,NaN,NB Webster Ave @ E 1,0,<NA>,"NB Webster Ave @ E 1, New York, NY"
86,5604553130,NaN,EB E Fordham Rd @ Va,0,<NA>,"EB E Fordham Rd @ Va, New York, NY"
158,5135876807,NaN,NARROWS RD S (E/B) @,0,<NA>,"NARROWS RD S (E/B) @, New York, NY"
162,5604648383,NaN,EB W 178th St @ Fort,0,<NA>,"EB W 178th St @ Fort, New York, NY"
414,1493254200,1225,FULTON ST,7,<NA>,"1225 FULTON ST, New York, NY"
800,1495141895,194-176,E 128 ST,25,<NA>,"194-176 E 128 ST, New York, NY"
1192,5604569653,NaN,EB E 149th St @ Broo,0,<NA>,"EB E 149th St @ Broo, New York, NY"
1225,5604663669,NaN,NB Webster Ave @ E 1,0,<NA>,"NB Webster Ave @ E 1, New York, NY"
1232,5604617738,NaN,EB E Fordham Rd @ El,0,<NA>,"EB E Fordham Rd @ El, New York, NY"
1266,5604639760,NaN,NB Webster Ave @ E 1,0,<NA>,"NB Webster Ave @ E 1, New York, NY"


In [ ]:
"""missing_boroughs["matched_address"] = (
    missing_boroughs["matched_address"]
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna(missing_boroughs["lookup_address"])
)"""

<>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\jayson.coker\AppData\Local\Temp\ipykernel_19784\2264107856.py:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  .replace(r"^\s*$", pd.NA, regex=True)


' missing_boroughs["matched_address"] = (\n    missing_boroughs["matched_address"]\n    .replace(r"^\\s*$", pd.NA, regex=True)\n    .fillna(missing_boroughs["lookup_address"])\n) '

## Displaying the output so that I can verify that the data copied over like it should.


In [ ]:
"""display(
    missing_boroughs[
        ["lookup_address", "matched_address", "suggested_borough", "lookup_status"]
    ]
)"""

' display(\n    missing_boroughs[\n        ["lookup_address", "matched_address", "suggested_borough", "lookup_status"]\n    ]\n) '

In [ ]:
GEOCODER_URL = "https://geosearch.planninglabs.nyc/v2/search"
session = requests.Session()

# Configuration for retry logic
MAX_GEOCODING_RETRIES = 3
RETRY_BACKOFF_FACTOR = 2  # Exponential backoff multiplier


def lookup_nyc_address(address: str, max_retries: int = MAX_GEOCODING_RETRIES) -> dict:
    """Look up an address using NYC's geocoder API with retry logic.
    
    Args:
        address: Address string to look up
        max_retries: Maximum number of retry attempts
    
    Returns:
        Dictionary with geocoding results
    """
    # Avoid unreliable searches that only contain a street name
    first_part = address.split(",")[0].strip()
    if not first_part or not any(character.isdigit() for character in first_part):
        return {
            "suggested_borough": None,
            "confidence": None,
            "matched_address": None,
            "lookup_status": "insufficient address",
        }

    last_error = None
    
    for attempt in range(max_retries):
        try:
            response = session.get(
                GEOCODER_URL,
                params={"text": address},
                timeout=15,
            )
            response.raise_for_status()

            features = response.json().get("features", [])

            if not features:
                return {
                    "suggested_borough": None,
                    "confidence": None,
                    "matched_address": None,
                    "lookup_status": "no match",
                }

            properties = features[0]["properties"]

            return {
                "suggested_borough": properties.get("borough"),
                "confidence": properties.get("confidence"),
                "matched_address": properties.get("label"),
                "lookup_status": "matched",
            }

        except RequestException as error:
            last_error = error
            if attempt < max_retries - 1:
                # Exponential backoff: 1s, 2s, 4s, etc.
                wait_time = RETRY_BACKOFF_FACTOR ** attempt
                print(f"Geocoding attempt {attempt + 1} failed for '{address}'. "
                      f"Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Geocoding failed for '{address}' after {max_retries} attempts: {error}")

    # All retries exhausted
    return {
        "suggested_borough": None,
        "confidence": None,
        "matched_address": None,
        "lookup_status": str(last_error) if last_error else "unknown error",
    }

In [ ]:
unique_addresses = missing_boroughs["lookup_address"].dropna().unique()

lookup_cache = {}

for number, address in enumerate(unique_addresses, start=1):
    lookup_cache[address] = lookup_nyc_address(address)

    if number % 25 == 0:
        print(f"Looked up {number:,} of {len(unique_addresses):,} addresses")

    time.sleep(0.1)

Looked up 25 of 301 addresses
Looked up 50 of 301 addresses
Looked up 75 of 301 addresses
Looked up 100 of 301 addresses
Looked up 125 of 301 addresses
Looked up 150 of 301 addresses
Looked up 175 of 301 addresses
Looked up 200 of 301 addresses
Looked up 225 of 301 addresses
Looked up 250 of 301 addresses
Looked up 275 of 301 addresses
Looked up 300 of 301 addresses


In [ ]:
lookup_results = missing_boroughs["lookup_address"].map(lookup_cache).apply(pd.Series)

# Assignment works whether these columns already exist or not.
missing_boroughs[lookup_results.columns] = lookup_results

display(
    missing_boroughs[
        [
            "summons_number",
            "lookup_address",
            "suggested_borough",
            "confidence",
            "matched_address",
            "lookup_status",
        ]
    ].sort_values("confidence", ascending=False)
)

,summons_number,lookup_address,suggested_borough,confidence,matched_address,lookup_status
5714,1472122045,"298 CLASSON, New York, NY",Brooklyn,1.0,"298 CLASSON AVENUE, Brooklyn, NY, USA",matched
11322,1495224545,"310 WEST 55, New York, NY",Manhattan,1.0,"310 WEST 55 STREET, New York, NY, USA",matched
18643,1490666930,"1601 BWAY, New York, NY",Manhattan,1.0,"1601 B'WAY, New York, NY, USA",matched
16192,1493899788,"1190 E 96, New York, NY",Brooklyn,1.0,"1190 EAST 96 STREET, Brooklyn, NY, USA",matched
15709,1495626271,"1448 , New York, NY",Manhattan,1.0,"1448 BROADWAY, New York, NY, USA",matched
...,...,...,...,...,...,...
99367,5604542386,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,NaN,no match
99475,5604600039,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,NaN,no match
99567,5604563250,"EB W Fordham Rd @ Gr, New York, NY",NaN,NaN,NaN,insufficient address
99608,5604617301,"EB E Fordham Rd @ Wa, New York, NY",NaN,NaN,NaN,insufficient address


In [ ]:
missing_boroughs["address_for_review"] = (
    missing_boroughs["matched_address"]
    .astype("string")
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna(missing_boroughs["lookup_address"].astype("string"))
)

display(
    missing_boroughs[
        [
            "lookup_address",
            "matched_address",
            "address_for_review",
            "suggested_borough",
            "confidence",
            "lookup_status",
        ]
    ]
)

,lookup_address,matched_address,address_for_review,suggested_borough,confidence,lookup_status
7,"NB Webster Ave @ E 1, New York, NY",NaN,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,no match
86,"EB E Fordham Rd @ Va, New York, NY",NaN,"EB E Fordham Rd @ Va, New York, NY",NaN,NaN,insufficient address
158,"NARROWS RD S (E/B) @, New York, NY",NaN,"NARROWS RD S (E/B) @, New York, NY",NaN,NaN,insufficient address
162,"EB W 178th St @ Fort, New York, NY",NaN,"EB W 178th St @ Fort, New York, NY",NaN,NaN,no match
414,"1225 FULTON ST, New York, NY","1225 FULTON STREET, Brooklyn, NY, USA","1225 FULTON STREET, Brooklyn, NY, USA",Brooklyn,0.8,matched
...,...,...,...,...,...,...
99375,"79 TYSEN ST, New York, NY","79 TYSEN STREET, Staten Island, NY, USA","79 TYSEN STREET, Staten Island, NY, USA",Staten Island,0.8,matched
99475,"NB Webster Ave @ E 1, New York, NY",NaN,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,no match
99567,"EB W Fordham Rd @ Gr, New York, NY",NaN,"EB W Fordham Rd @ Gr, New York, NY",NaN,NaN,insufficient address
99608,"EB E Fordham Rd @ Wa, New York, NY",NaN,"EB E Fordham Rd @ Wa, New York, NY",NaN,NaN,insufficient address


In [ ]:
display(
    missing_boroughs[
        ["lookup_address", "matched_address", "suggested_borough", "lookup_status"]
    ]
)

,lookup_address,matched_address,suggested_borough,lookup_status
7,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,no match
86,"EB E Fordham Rd @ Va, New York, NY",NaN,NaN,insufficient address
158,"NARROWS RD S (E/B) @, New York, NY",NaN,NaN,insufficient address
162,"EB W 178th St @ Fort, New York, NY",NaN,NaN,no match
414,"1225 FULTON ST, New York, NY","1225 FULTON STREET, Brooklyn, NY, USA",Brooklyn,matched
...,...,...,...,...
99375,"79 TYSEN ST, New York, NY","79 TYSEN STREET, Staten Island, NY, USA",Staten Island,matched
99475,"NB Webster Ave @ E 1, New York, NY",NaN,NaN,no match
99567,"EB W Fordham Rd @ Gr, New York, NY",NaN,NaN,insufficient address
99608,"EB E Fordham Rd @ Wa, New York, NY",NaN,NaN,insufficient address


In [ ]:
high_confidence = missing_boroughs[
    missing_boroughs["suggested_borough"].notna()
    & missing_boroughs["confidence"].ge(0.60)
]

print(f"Missing borough rows: {len(missing_boroughs):,}")
print(f"High-confidence matches: {len(high_confidence):,}")

display(high_confidence.head(30))

Missing borough rows: 844
High-confidence matches: 221


,summons_number,plate_id,registration_state,plate_type,issue_date,violation_code,vehicle_body_type,vehicle_make,issuing_agency,violation_precinct,issuer_precinct,violation_time,violation_county,street_name,vehicle_color,vehicle_year,violation_description,issue_year,issue_month,issue_day_of_week,issue_day_name,borough,house_number,lookup_address,suggested_borough,confidence,matched_address,lookup_status,address_for_review
414,1493254200,JJG,NY,PAS,2024-06-08,46,SDN,ME/BE,P,7,79,1000P,<NA>,FULTON ST,WH,2024,<NA>,2024,6,5,Saturday,NaN,1225,"1225 FULTON ST, New York, NY",Brooklyn,0.8,"1225 FULTON STREET, Brooklyn, NY, USA",matched,"1225 FULTON STREET, Brooklyn, NY, USA"
800,1495141895,LJW5169,NY,PAS,2024-06-11,50,SUBN,VOLKS,P,25,25,1227P,<NA>,E 128 ST,GREY,2024,<NA>,2024,6,1,Tuesday,NaN,194-176,"194-176 E 128 ST, New York, NY",Manhattan,0.8,"176 EAST 128 STREET, New York, NY, USA",matched,"176 EAST 128 STREET, New York, NY, USA"
2629,1493238190,DNJ3895,FL,PAS,2024-06-17,46,SDN,TOYOT,P,83,83,1158A,<NA>,SUYDAM ST,RD,0,<NA>,2024,6,0,Monday,NaN,400,"400 SUYDAM ST, New York, NY",Brooklyn,0.8,"400 SUYDAM STREET, Brooklyn, NY, USA",matched,"400 SUYDAM STREET, Brooklyn, NY, USA"
3251,1489829878,G5595L,FL,PAS,2024-07-07,98,SUBN,VOLKS,P,115,115,1208A,<NA>,32 AVE,WH,2003,<NA>,2024,7,6,Sunday,NaN,86-11,"86-11 32 AVE, New York, NY",Queens,0.8,"86-11 32 AVENUE, East Elmhurst, NY, USA",matched,"86-11 32 AVENUE, East Elmhurst, NY, USA"
3878,1494115001,BLANKPLATE,99,999,2024-06-22,74,SDN,ME/BE,P,40,40,0940P,<NA>,EAGLE AVE,<NA>,0,<NA>,2024,6,5,Saturday,NaN,662,"662 EAGLE AVE, New York, NY",Bronx,0.8,"662 EAGLE AVENUE, Bronx, NY, USA",matched,"662 EAGLE AVENUE, Bronx, NY, USA"
3939,1495300067,KMP1156,NY,PAS,2024-06-14,37,SUBN,CHEVR,P,103,103,0745P,<NA>,149 STREET,<NA>,0,<NA>,2024,6,4,Friday,NaN,NaN,"149 STREET, New York, NY",Manhattan,0.8,"206 WEST 149 STREET, New York, NY, USA",matched,"206 WEST 149 STREET, New York, NY, USA"
4123,1491618358,LDG1038,NY,PAS,2024-06-28,40,SDN,VOLKS,8,43,0,0936P,<NA>,E TREMONT AVE,GREY,0,<NA>,2024,6,4,Friday,NaN,2130,"2130 E TREMONT AVE, New York, NY",Bronx,0.8,"2130 EAST TREMONT AVENUE, Bronx, NY, USA",matched,"2130 EAST TREMONT AVENUE, Bronx, NY, USA"
5159,1496076291,52073NE,NY,COM,2024-06-21,78,DELV,INTER,P,114,114,0940P,<NA>,46 ST,WH,2025,<NA>,2024,6,4,Friday,NaN,21-18,"21-18 46 ST, New York, NY",Queens,0.8,"21-18 46 STREET, Astoria, NY, USA",matched,"21-18 46 STREET, Astoria, NY, USA"
5592,1485527727,FBC8209,NY,PAS,2024-07-10,98,<NA>,<NA>,P,0,73,0140A,<NA>,MOTHER GASTON AVE,<NA>,0,<NA>,2024,7,2,Wednesday,NaN,22,"22 MOTHER GASTON AVE, New York, NY",Brooklyn,0.8,"22 MOTHER GASTON BOULEVARD, Brooklyn, NY, USA",matched,"22 MOTHER GASTON BOULEVARD, Brooklyn, NY, USA"
5714,1472122045,XNYY32,NJ,PAS,2024-07-03,46,VAN,RAM,P,88,88,1134A,<NA>,CLASSON,WHITE,0,<NA>,2024,7,2,Wednesday,NaN,298,"298 CLASSON, New York, NY",Brooklyn,1.0,"298 CLASSON AVENUE, Brooklyn, NY, USA",matched,"298 CLASSON AVENUE, Brooklyn, NY, USA"


In [ ]:
clean_sample.loc[high_confidence.index, "borough"] = high_confidence[
    "suggested_borough"
]

clean_sample["borough"].value_counts(dropna=False)

borough
Queens           30042
Brooklyn         26738
Manhattan        23624
Bronx            13997
Staten Island     4968
NaN                623
Name: count, dtype: int64

In [ ]:
# Input and output file locations
raw_file = Path("../data/raw/nycparking2025.csv")
output_file = Path("../data/processed/parking_locations.csv")

# Columns to extract
location_columns = [
    "Summons Number",
    "Issue Date",
    "Street Code1",
    "Street Code2",
    "Street Code3",
    "Violation Location",
    "Violation Precinct",
    "Violation County",
    "Street Name",
]

# Create the output directory if necessary
output_file.parent.mkdir(parents=True, exist_ok=True)

chunk_size = 100_000
first_chunk = True
total_rows = 0

for chunk_number, location_df in enumerate(
    pd.read_csv(
        raw_file,
        usecols=location_columns,
        dtype="string",
        chunksize=chunk_size,
        low_memory=False,
    ),
    start=1,
):
    # Write the first chunk with column headers.
    # Later chunks are appended without repeating the headers.
    location_df.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    first_chunk = False
    total_rows += len(location_df)

    print(
        f"Chunk {chunk_number}: "
        f"{len(location_df):,} rows saved; "
        f"{total_rows:,} total rows"
    )

print(f"\nFinished saving {total_rows:,} rows to:")
print(output_file.resolve())

# Preview the final chunk processed
display(location_df.head())

In [ ]:
sample[["Street Code1", "Street Code2", "Street Code3"]].head(500)

In [ ]:
borough_codes = {
    "NY": "1",  # Manhattan
    "MN": "1",
    "BX": "2",  # Bronx
    "K": "3",  # Brooklyn
    "BK": "3",
    "Q": "4",  # Queens
    "QN": "4",
    "R": "5",  # Staten Island
    "ST": "5",
}

sample["borough_code"] = sample["Violation County"].map(borough_codes)

for column in ["Street Code1", "Street Code2", "Street Code3"]:
    sample[f"{column}_B5SC"] = (
        sample["borough_code"] + sample[column].astype("string").str.zfill(5)
    ).where(sample[column] != "0")

sample[
    [
        "Street Name",
        "Violation County",
        "Street Code1_B5SC",
        "Street Code2_B5SC",
        "Street Code3_B5SC",
    ]
].head(20)

In [ ]:
BOROUGH_CODES = {
    "NY": "1",
    "MN": "1",
    "BX": "2",
    "K": "3",
    "BK": "3",
    "Q": "4",
    "QN": "4",
    "R": "5",
    "ST": "5",
}


def format_goat_street_codes(row):
    """Return valid Street Code1–3 values formatted as B5SC codes."""
    borough = BOROUGH_CODES.get(str(row["Violation County"]).strip().upper())

    if borough is None:
        return []

    goat_codes = []

    for column in ["Street Code1", "Street Code2", "Street Code3"]:
        value = pd.to_numeric(row[column], errors="coerce")

        # Skip missing values and zero.
        if pd.isna(value) or value == 0:
            continue

        # Borough digit + five-digit street code.
        goat_codes.append(f"{borough}{int(value):05d}")

    return goat_codes

In [ ]:
sample["goat_street_codes"] = sample.apply(format_goat_street_codes, axis=1)

sample[
    [
        "Violation County",
        "Street Code1",
        "Street Code2",
        "Street Code3",
        "goat_street_codes",
    ]
].head(20)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/geosupport_borough_matches.csv", dtype=str).fillna(
    ""
)

df[df["status"] == "ambiguous"].to_csv("ambiguous.csv", index=False)
df[df["status"] == "review"].to_csv("review.csv", index=False)
df[df["status"] == "unmatched"].to_csv("unmatched.csv", index=False)

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

# Works from either the project folder or notebooks folder
PROJECT_ROOT = Path.cwd().resolve()

while (
    not (PROJECT_ROOT / "pyproject.toml").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATABASE_FILE = PROJECT_ROOT / "data/database/nyc_parking.sqlite"

GEOSUPPORT_FILE = PROJECT_ROOT / "data/processed/geosupport_borough_matches.csv"

AMBIGUOUS_FILE = PROJECT_ROOT / "data/processed/ambiguous_reprocessed.csv"

# Optional CSV containing boroughs you personally reviewed and approved
MANUAL_FILE = PROJECT_ROOT / "data/processed/manual_borough_recovery.csv"

# Final combined recovery CSV
RECOVERY_FILE = PROJECT_ROOT / "data/processed/borough_recovery.csv"

print("Project:", PROJECT_ROOT)
print("Database:", DATABASE_FILE)

In [ ]:
required_files = [
    DATABASE_FILE,
    GEOSUPPORT_FILE,
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(file)

print("Required files found.")
print("Ambiguous results found:", AMBIGUOUS_FILE.exists())
print("Manual review file found:", MANUAL_FILE.exists())

In [ ]:
VALID_BOROUGHS = {
    "Bronx",
    "Brooklyn",
    "Manhattan",
    "Queens",
    "Staten Island",
}

geosupport_results = pd.read_csv(
    GEOSUPPORT_FILE,
    dtype="string",
)

accepted_results = geosupport_results.loc[
    geosupport_results["status"].eq("accepted")
    & geosupport_results["suggested_borough"].isin(VALID_BOROUGHS),
    [
        "summons_number",
        "suggested_borough",
        "confidence",
        "validation_method",
    ],
].copy()

accepted_results = accepted_results.rename(
    columns={
        "suggested_borough": "recovered_borough",
        "validation_method": "recovery_method",
    }
)

accepted_results["source_file"] = GEOSUPPORT_FILE.name

print(f"Accepted automatic results: {len(accepted_results):,}")
accepted_results.head()

In [ ]:
resolved_ambiguous = pd.DataFrame(
    columns=[
        "summons_number",
        "recovered_borough",
        "confidence",
        "recovery_method",
        "source_file",
    ]
)

if AMBIGUOUS_FILE.exists():
    ambiguous_results = pd.read_csv(
        AMBIGUOUS_FILE,
        dtype="string",
    )

    resolved_ambiguous = ambiguous_results.loc[
        ambiguous_results["review_method"].eq("parsed_intersection_function_2")
        & ambiguous_results["review_suggested_borough"].isin(VALID_BOROUGHS),
        [
            "summons_number",
            "review_suggested_borough",
            "review_confidence",
            "review_method",
        ],
    ].copy()

    resolved_ambiguous = resolved_ambiguous.rename(
        columns={
            "review_suggested_borough": "recovered_borough",
            "review_confidence": "confidence",
            "review_method": "recovery_method",
        }
    )

    resolved_ambiguous["source_file"] = AMBIGUOUS_FILE.name

print(f"Resolved ambiguous results: {len(resolved_ambiguous):,}")

In [ ]:
if not MANUAL_FILE.exists():
    manual_template = pd.DataFrame(
        columns=[
            "summons_number",
            "recovered_borough",
        ]
    )

    manual_template.to_csv(MANUAL_FILE, index=False)
    print(f"Created manual-review template:\n{MANUAL_FILE}")

In [ ]:
manual_results = pd.read_csv(
    MANUAL_FILE,
    dtype="string",
)

manual_results = manual_results[
    manual_results["recovered_borough"].isin(VALID_BOROUGHS)
].copy()

manual_results["confidence"] = "manual"
manual_results["recovery_method"] = "manual_review"
manual_results["source_file"] = MANUAL_FILE.name

manual_results = manual_results[
    [
        "summons_number",
        "recovered_borough",
        "confidence",
        "recovery_method",
        "source_file",
    ]
]

print(f"Manual results: {len(manual_results):,}")
manual_results.head()

In [ ]:
recovery = pd.concat(
    [
        accepted_results,
        resolved_ambiguous,
        manual_results,
    ],
    ignore_index=True,
)

recovery["summons_number"] = pd.to_numeric(
    recovery["summons_number"],
    errors="coerce",
).astype("Int64")

recovery = recovery.dropna(subset=["summons_number", "recovered_borough"])

# Check for conflicting boroughs assigned to the same summons
conflicts = recovery.groupby("summons_number")["recovered_borough"].nunique()

conflicts = conflicts[conflicts > 1]

if not conflicts.empty:
    conflicting_rows = recovery[
        recovery["summons_number"].isin(conflicts.index)
    ].sort_values("summons_number")

    display(conflicting_rows)

    raise ValueError(f"{len(conflicts):,} summons numbers have conflicting boroughs")

# Manual results were added last, so they are retained for duplicates
recovery = recovery.drop_duplicates(
    subset=["summons_number"],
    keep="last",
)

recovery["summons_number"] = recovery["summons_number"].astype("int64")

print(f"Total approved recovery results: {len(recovery):,}")

recovery["recovered_borough"].value_counts()

In [ ]:
recovery.to_csv(
    RECOVERY_FILE,
    index=False,
)

print(f"Saved {len(recovery):,} rows to:")
print(RECOVERY_FILE)

In [ ]:
records = [
    (
        int(row.summons_number),
        row.recovered_borough,
        None if pd.isna(row.confidence) else str(row.confidence),
        None if pd.isna(row.recovery_method) else str(row.recovery_method),
        None if pd.isna(row.source_file) else str(row.source_file),
    )
    for row in recovery.itertuples(index=False)
]

with sqlite3.connect(DATABASE_FILE) as connection:
    connection.execute("PRAGMA foreign_keys = ON")

    connection.executescript("""
        CREATE TABLE IF NOT EXISTS borough_recovery (
            summons_number INTEGER PRIMARY KEY,
            recovered_borough TEXT NOT NULL,
            confidence TEXT,
            recovery_method TEXT,
            source_file TEXT,
            FOREIGN KEY (summons_number)
                REFERENCES parking_violations(summons_number)
        );

        DELETE FROM borough_recovery;

        DROP TABLE IF EXISTS temp.recovery_stage;

        CREATE TEMP TABLE recovery_stage (
            summons_number INTEGER PRIMARY KEY,
            recovered_borough TEXT,
            confidence TEXT,
            recovery_method TEXT,
            source_file TEXT
        );
        """)

    connection.executemany(
        """
        INSERT INTO recovery_stage
        VALUES (?, ?, ?, ?, ?)
        """,
        records,
    )

    # Only insert summons numbers found in parking_violations
    connection.execute("""
        INSERT INTO borough_recovery
        SELECT
            s.summons_number,
            s.recovered_borough,
            s.confidence,
            s.recovery_method,
            s.source_file
        FROM recovery_stage AS s
        INNER JOIN parking_violations AS p
            ON p.summons_number = s.summons_number
        """)

    loaded_count = connection.execute(
        "SELECT COUNT(*) FROM borough_recovery"
    ).fetchone()[0]

print(f"Loaded {loaded_count:,} recovered boroughs into SQLite.")

In [ ]:
with sqlite3.connect(DATABASE_FILE) as connection:
    connection.executescript("""
        DROP VIEW IF EXISTS parking_analysis;

        CREATE VIEW parking_analysis AS
        SELECT
            p.*,

            COALESCE(
                NULLIF(TRIM(p.borough), ''),
                r.recovered_borough
            ) AS analysis_borough,

            CASE
                WHEN NULLIF(TRIM(p.borough), '') IS NOT NULL
                    THEN 'original'
                WHEN r.recovered_borough IS NOT NULL
                    THEN 'recovered'
                ELSE 'missing'
            END AS borough_source,

            r.confidence AS recovery_confidence,
            r.recovery_method,
            r.source_file AS recovery_source_file

        FROM parking_enriched AS p

        LEFT JOIN borough_recovery AS r
            ON r.summons_number = p.summons_number;
        """)

print("Created SQLite view: parking_analysis")

In [ ]:
with sqlite3.connect(DATABASE_FILE) as connection:
    validation = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS total_summons,

            SUM(
                CASE WHEN borough_source = 'original'
                THEN 1 ELSE 0 END
            ) AS original_boroughs,

            SUM(
                CASE WHEN borough_source = 'recovered'
                THEN 1 ELSE 0 END
            ) AS recovered_boroughs,

            SUM(
                CASE WHEN borough_source = 'missing'
                THEN 1 ELSE 0 END
            ) AS still_missing

        FROM parking_analysis
        """,
        connection,
    )

validation

In [ ]:
with sqlite3.connect(DATABASE_FILE) as connection:
    borough_analysis = pd.read_sql_query(
        """
        SELECT
            analysis_borough AS borough,
            borough_source,
            COUNT(*) AS summons_count
        FROM parking_analysis
        GROUP BY
            analysis_borough,
            borough_source
        ORDER BY summons_count DESC
        """,
        connection,
    )

borough_analysis

In [ ]:
""" camera_results = pd.read_csv(
    PROJECT_ROOT / "data/processed/geosupport_camera_description_accepted.csv",
    dtype="string",
)

camera_results = camera_results[
    [
        "summons_number",
        "suggested_borough",
        "confidence",
        "resolution_method",
    ]
].rename(
    columns={
        "suggested_borough": "recovered_borough",
        "resolution_method": "recovery_method",
    }
)

camera_results["source_file"] = "geosupport_camera_description_accepted.csv"

recovery = pd.concat(
    [
        accepted_results,
        resolved_ambiguous,
        manual_results,
        camera_results,
    ],
    ignore_index=True,
) """